## 1. Criação do Dataset de Pinça

In [ ]:
!pip install mediapipe==0.10.14

In [ ]:
import os
import glob
import cv2
import pandas as pd
import kagglehub

# a importacao explicita é feita para evitar problemas de carregamento de modulos no jupyter
import mediapipe.python.solutions.hands as mp_hands

# o detector de maos do mediapipe é inicializado
hand_detector = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.5
)

def calculate_normalized_distance(landmarks):
    """
    calculates the 3d distance between thumb and index finger,
    normalized by the overall hand size to prevent scale issues.
    """
    # as coordenadas espaciais das articulacoes sao mapeadas
    thumb_tip = landmarks.landmark[mp_hands.HandLandmark.THUMB_TIP]
    index_tip = landmarks.landmark[mp_hands.HandLandmark.INDEX_FINGER_TIP]
    wrist = landmarks.landmark[mp_hands.HandLandmark.WRIST]
    middle_mcp = landmarks.landmark[mp_hands.HandLandmark.MIDDLE_FINGER_MCP]

    # a distancia euclidiana 3d entre o polegar e o indicador é calculada
    pinch_distance = (
        (thumb_tip.x - index_tip.x)**2 +
        (thumb_tip.y - index_tip.y)**2 +
        (thumb_tip.z - index_tip.z)**2
    )**0.5

    # o tamanho base da mao é calculado usando o pulso e o dedo medio
    hand_size = (
        (wrist.x - middle_mcp.x)**2 +
        (wrist.y - middle_mcp.y)**2 +
        (wrist.z - middle_mcp.z)**2
    )**0.5

    # a feature final normalizada é retornada
    return pinch_distance / hand_size if hand_size > 0 else 0

def build_tabular_dataset():
    """
    downloads the image dataset and converts it to a tabular format with features.
    """
    print("iniciando o download/leitura do dataset no kaggle...")
    # o dataset original é baixado via kagglehub
    dataset_path = kagglehub.dataset_download("gti-upm/leapgestrecog")
    print(f"dataset carregado com sucesso no diretório: {dataset_path}")

    # as listas de dados sao inicializadas
    extracted_features = []
    target_labels = []

    # as classes do problema de classificacao binaria sao definidas
    # 1 representa a pinça (ok) e 0 representa a mao aberta (palm)
    class_mapping = {'07_ok': 1, '01_palm': 0}

    # os diretorios de cada classe sao iterados
    for class_name, label in class_mapping.items():
        print(f"\nbuscando imagens para a classe '{class_name}'...")
        # o caminho relativo para todas as imagens da classe atual é construído
        search_pattern = os.path.join(dataset_path, '**', class_name, '*.png')
        image_paths = glob.glob(search_pattern, recursive=True)

        total_images = len(image_paths)
        print(f"foram encontradas {total_images} imagens. iniciando o processamento...")

        for index, img_path in enumerate(image_paths):
            # a imagem é carregada e convertida para o padrao rgb
            image_bgr = cv2.imread(img_path)
            if image_bgr is None:
                continue

            image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

            # o processamento da inferencia da mao é executado
            detection_results = hand_detector.process(image_rgb)

            # é verificado se os landmarks foram encontrados na imagem atual
            if detection_results.multi_hand_landmarks:
                # as features da primeira mao detectada sao extraídas e armazenadas
                first_hand_landmarks = detection_results.multi_hand_landmarks[0]
                feature_value = calculate_normalized_distance(first_hand_landmarks)

                extracted_features.append(feature_value)
                target_labels.append(label)

            # o log de progresso é impresso a cada 100 imagens processadas
            if (index + 1) % 100 == 0:
                print(f"[{class_name}] progresso: {index + 1}/{total_images} imagens analisadas...")

    print("\nprocessamento das imagens concluído. gerando o arquivo csv...")
    # o dataframe tabular é estruturado
    dataset_df = pd.DataFrame({
        'feature_normalized_distance': extracted_features,
        'label': target_labels
    })

    # o arquivo csv é persistido no disco
    dataset_df.to_csv('/content/drive/MyDrive/cte-ia/dataset-hand-pinch/pinch_features_dataset.csv', index=False)
    print("dataset tabular 'pinch_features_dataset.csv' gerado com sucesso!")

    return dataset_df

if __name__ == "__main__":
    build_tabular_dataset()

iniciando o download/leitura do dataset no kaggle...
Using Colab cache for faster access to the 'leapgestrecog' dataset.
dataset carregado com sucesso no diretório: /kaggle/input/leapgestrecog

buscando imagens para a classe '07_ok'...
foram encontradas 4000 imagens. iniciando o processamento...
[07_ok] progresso: 100/4000 imagens analisadas...
[07_ok] progresso: 200/4000 imagens analisadas...
[07_ok] progresso: 300/4000 imagens analisadas...
[07_ok] progresso: 400/4000 imagens analisadas...
[07_ok] progresso: 500/4000 imagens analisadas...
[07_ok] progresso: 600/4000 imagens analisadas...
[07_ok] progresso: 700/4000 imagens analisadas...
[07_ok] progresso: 800/4000 imagens analisadas...
[07_ok] progresso: 900/4000 imagens analisadas...
[07_ok] progresso: 1000/4000 imagens analisadas...
[07_ok] progresso: 1100/4000 imagens analisadas...
[07_ok] progresso: 1200/4000 imagens analisadas...
[07_ok] progresso: 1300/4000 imagens analisadas...
[07_ok] progresso: 1400/4000 imagens analisadas..

## 2. Treinamento do Modelo de Detecção

In [ ]:
!pip install scikit-learn m2cgen

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.2/92.2 kB 3.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score
import m2cgen as m2c

def train_and_export_model():
    # o caminho do dataset gerado anteriormente é definido
    dataset_path = '/content/drive/MyDrive/cte-ia/dataset-hand-pinch/pinch_features_dataset.csv'

    # os dados são carregados em um dataframe pandas
    dataset = pd.read_csv(dataset_path)

    # as features de entrada e a variável alvo são separadas
    features = dataset[['feature_normalized_distance']]
    labels = dataset['label']

    # o conjunto de dados é dividido em treino (80%) e teste (20%)
    x_train, x_test, y_train, y_test = train_test_split(
        features, labels, test_size=0.2, random_state=42
    )

    # o modelo classificador base é instanciado
    base_model = DecisionTreeClassifier(random_state=42)

    # a grade de hiperparâmetros para o fine-tuning é definida
    hyperparameters = {
        'max_depth': [2, 3, 4, 5],
        'min_samples_split': [2, 5, 10],
        'criterion': ['gini', 'entropy']
    }

    # a busca exaustiva com validação cruzada (5 folds) é configurada
    grid_search = GridSearchCV(base_model, hyperparameters, cv=5, scoring='accuracy')

    print("o treinamento e a busca de hiperparâmetros foram iniciados...")

    # o modelo é treinado testando todas as combinações
    grid_search.fit(x_train, y_train)

    # o estimador com o melhor desempenho é extraído
    best_model = grid_search.best_estimator_

    print(f"os melhores hiperparâmetros encontrados foram: {grid_search.best_params_}")

    # as predições são geradas usando o conjunto de teste separado
    predictions = best_model.predict(x_test)

    # as métricas de avaliação são computadas
    accuracy = accuracy_score(y_test, predictions)
    report = classification_report(y_test, predictions)

    print(f"\nacurácia no teste: {accuracy:.4f}")
    print("\nrelatório de classificação:\n", report)

    # a árvore de decisão treinada é transpilada para código javascript puro
    javascript_code = m2c.export_to_javascript(best_model)

    # o arquivo javascript final é salvo no disco
    output_js_path = '/content/drive/MyDrive/cte-ia/dataset-hand-pinch/pinch_model.js'

    with open(output_js_path, 'w') as js_file:
        js_file.write(javascript_code)

    print(f"\nmodelo exportado com sucesso para: {output_js_path}")

if __name__ == "__main__":
    train_and_export_model()

o treinamento e a busca de hiperparâmetros foram iniciados...
os melhores hiperparâmetros encontrados foram: {'criterion': 'gini', 'max_depth': 4, 'min_samples_split': 2}

acurácia no teste: 0.8182

relatório de classificação:
               precision    recall  f1-score   support

           0       0.79      0.89      0.84       722
           1       0.87      0.73      0.79       664

    accuracy                           0.82      1386
   macro avg       0.83      0.81      0.82      1386
weighted avg       0.82      0.82      0.82      1386


modelo exportado com sucesso para: /content/drive/MyDrive/cte-ia/dataset-hand-pinch/pinch_model.js
